# 01 - EDA\n\nExploracao inicial do dataset HVFHV usando Spark SQL. Todas as transformacoes analiticas ficam em `spark.sql()`; pandas aparece apenas depois de agregacoes pequenas para visualizacao.

In [ ]:
from pyspark.sql import SparkSession\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nSEED = 42\nDATA_GLOB = '/data/fhvhv_tripdata_*.parquet'\nLOOKUP_PATH = '/data/taxi_zone_lookup.csv'\n\nspark = (SparkSession.builder\n    .appName('nyc-rideshare-eda')\n    .master('spark://spark-master:7077')\n    .config('spark.executor.memory', '3g')\n    .config('spark.driver.memory', '4g')\n    .config('spark.sql.shuffle.partitions', '200')\n    .getOrCreate())\n\nspark.sparkContext.setLogLevel('WARN')\nsns.set_theme(style='whitegrid')

In [ ]:
trips_df = spark.read.parquet(DATA_GLOB)\nzones_df = (spark.read\n    .option('header', True)\n    .option('inferSchema', True)\n    .csv(LOOKUP_PATH))\n\ntrips_df.createOrReplaceTempView('trips_bronze')\nzones_df.createOrReplaceTempView('taxi_zone_lookup')\n\nspark.sql('SELECT COUNT(*) AS total_rows FROM trips_bronze').show()

In [ ]:
spark.sql("""\nSELECT\n    MIN(request_datetime) AS min_request_datetime,\n    MAX(dropoff_datetime) AS max_dropoff_datetime,\n    COUNT(DISTINCT DATE(pickup_datetime)) AS active_days,\n    COUNT(*) AS total_rows\nFROM trips_bronze\n""").show(truncate=False)\n\nspark.sql("""\nSELECT\n    DATE_FORMAT(pickup_datetime, 'yyyy-MM') AS pickup_month,\n    hvfhs_license_num,\n    COUNT(*) AS trips\nFROM trips_bronze\nGROUP BY 1, 2\nORDER BY 1, 2\n""").show(100, truncate=False)

In [ ]:
spark.sql("""\nSELECT 'on_scene_datetime' AS column_name, SUM(CASE WHEN on_scene_datetime IS NULL THEN 1 ELSE 0 END) AS null_rows, COUNT(*) AS total_rows FROM trips_bronze\nUNION ALL\nSELECT 'originating_base_num', SUM(CASE WHEN originating_base_num IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM trips_bronze\nUNION ALL\nSELECT 'tips', SUM(CASE WHEN tips IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM trips_bronze\nUNION ALL\nSELECT 'airport_fee', SUM(CASE WHEN airport_fee IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM trips_bronze\nUNION ALL\nSELECT 'driver_pay', SUM(CASE WHEN driver_pay IS NULL THEN 1 ELSE 0 END), COUNT(*) FROM trips_bronze\n""").show(truncate=False)

In [ ]:
spark.sql("""\nSELECT\n    PERCENTILE_APPROX(trip_miles, ARRAY(0.01, 0.5, 0.99)) AS trip_miles_pct,\n    PERCENTILE_APPROX(trip_time, ARRAY(0.01, 0.5, 0.99)) AS trip_time_pct,\n    PERCENTILE_APPROX(base_passenger_fare, ARRAY(0.01, 0.5, 0.99)) AS fare_pct,\n    PERCENTILE_APPROX(driver_pay, ARRAY(0.01, 0.5, 0.99)) AS driver_pay_pct\nFROM trips_bronze\nWHERE trip_miles IS NOT NULL\n  AND trip_time IS NOT NULL\n  AND base_passenger_fare IS NOT NULL\n""").show(truncate=False)\n\nspark.sql("""\nSELECT\n    COUNT(*) AS invalid_temporal_rows\nFROM trips_bronze\nWHERE pickup_datetime < request_datetime\n   OR dropoff_datetime <= pickup_datetime\n""").show()

In [ ]:
hourly_pdf = spark.sql("""\nSELECT\n    HOUR(pickup_datetime) AS pickup_hour,\n    COUNT(*) AS trips\nFROM trips_bronze\nGROUP BY 1\nORDER BY 1\n""").toPandas()\n\nax = sns.lineplot(data=hourly_pdf, x='pickup_hour', y='trips', marker='o')\nax.set(title='Volume de corridas por hora', xlabel='Hora do dia', ylabel='Corridas')\nplt.show()

In [ ]:
top_pickups_pdf = spark.sql("""\nSELECT\n    z.Zone,\n    z.Borough,\n    COUNT(*) AS trips\nFROM trips_bronze t\nLEFT JOIN taxi_zone_lookup z\n  ON t.PULocationID = z.LocationID\nWHERE t.PULocationID NOT IN (264, 265)\nGROUP BY 1, 2\nORDER BY trips DESC\nLIMIT 10\n""").toPandas()\n\nax = sns.barplot(data=top_pickups_pdf, x='trips', y='Zone', hue='Borough', dodge=False)\nax.set(title='Top 10 zonas de pickup', xlabel='Corridas', ylabel='Zona')\nplt.show()

In [ ]:
borough_od_pdf = spark.sql("""\nSELECT\n    pu.Borough AS pu_borough,\n    do.Borough AS do_borough,\n    COUNT(*) AS trips\nFROM trips_bronze t\nLEFT JOIN taxi_zone_lookup pu\n  ON t.PULocationID = pu.LocationID\nLEFT JOIN taxi_zone_lookup do\n  ON t.DOLocationID = do.LocationID\nWHERE t.PULocationID NOT IN (264, 265)\n  AND t.DOLocationID NOT IN (264, 265)\nGROUP BY 1, 2\n""").toPandas()\n\npivot = borough_od_pdf.pivot(index='pu_borough', columns='do_borough', values='trips').fillna(0)\nplt.figure(figsize=(10, 6))\nsns.heatmap(pivot, cmap='Blues', fmt='.0f')\nplt.title('Matriz origem-destino entre boroughs')\nplt.show()

In [ ]:
spark.sql("""\nSELECT\n    corr(trip_miles, base_passenger_fare) AS corr_miles_fare,\n    corr(trip_time, base_passenger_fare) AS corr_time_fare,\n    corr(driver_pay, base_passenger_fare) AS corr_driver_pay_fare,\n    corr(tips, base_passenger_fare) AS corr_tips_fare\nFROM trips_bronze\nWHERE trip_miles IS NOT NULL\n  AND trip_time IS NOT NULL\n  AND base_passenger_fare IS NOT NULL\n""").show(truncate=False)

## Observacoes para o log de decisoes\n\n- Quantificar nulos de `on_scene_datetime` por operadora antes de descartar esse sinal de modelagem.\n- Documentar a participacao dominante da Uber para evitar comparacoes brutas entre operadoras.\n- Registrar o percentual removido por inconsistencias temporais e por velocidades extremas no notebook de preprocessing.

In [ ]:
spark.stop()